# RGD: Interactive Component Walkthrough
This notebook demonstrates and tests each component of the **Research Gap Discovery** pipeline individually.

In [1]:
import sys
import os
# Ensure src is discoverable
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import yaml
from dotenv import load_dotenv

load_dotenv("../.env")
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
print("Project:", config["project"]["name"])

Project: Research Gap Discovery


## Phase 1: Literature Acquisition & Deduplication

In [2]:
from src.phase1_acquisition.openalex_fetch import OpenAlexFetcher
from src.phase1_acquisition.deduplication import CorpusDeduplicator

fetcher = OpenAlexFetcher()
seed_papers = fetcher.fetch_papers_by_query("physics informed neural networks", max_results=10)
df_raw = fetcher.to_dataframe(seed_papers)
print(f"Fetched {len(df_raw)} seed papers.")

deduplicator = CorpusDeduplicator(threshold=0.85)
df_clean, removed = deduplicator.deduplicate(df_raw)
print(f"Retained {len(df_clean)} papers (Removed {removed} duplicates).")

Fetched 10 seed papers.
Retained 10 papers (Removed 0 duplicates).


In [3]:
df_clean

,id,doi,title,publication_year,cited_by_count,abstract,authors,concepts,referenced_works,related_works,normalized_title
0,https://openalex.org/W2989521711,https://doi.org/10.1109/pesgm41954.2020.9282004,Physics-Informed Neural Networks for Power Sys...,2020,338,"This paper introduces for the first time, to o...","[George S. Misyris, Andreas Venzke, Spyros Cha...","[Artificial neural network, Inertia, Computer ...","[https://openalex.org/W1513238352, https://ope...","[https://openalex.org/W4386018708, https://ope...",physics-informed neural networks for power sys...
1,https://openalex.org/W4308979219,https://doi.org/10.2139/ssrn.4086448,Self-Adaptive Physics-Informed Neural Networks,2022,718,,"[Levi D. McClenny, Ulisses Braga-Neto]","[Artificial neural network, Statistical physic...","[https://openalex.org/W1870542990, https://ope...","[https://openalex.org/W4287671439, https://ope...",self-adaptive physics-informed neural networks
2,https://openalex.org/W2900501930,https://doi.org/10.1137/18m1229845,fPINNs: Fractional Physics-Informed Neural Net...,2019,1109,"Physics-informed neural networks (PINNs), intr...","[Guofei Pang, Lu Lu, George Em Karniadakis]","[Discretization, Forcing (mathematics), Fracti...","[https://openalex.org/W617206983, https://open...","[https://openalex.org/W7165002367, https://ope...",fpinns: fractional physics-informed neural net...
3,https://openalex.org/W4226480439,https://doi.org/10.1016/j.cma.2022.114790,Physics informed neural networks for continuum...,2022,236,,"[Alexander Henkes, Henning Wessels, Rolf Mahnken]","[Ansatz, Artificial neural network, Nonlinear ...","[https://openalex.org/W313791183, https://open...","[https://openalex.org/W3188026634, https://ope...",physics informed neural networks for continuum...
4,https://openalex.org/W4220717841,https://doi.org/10.1007/s10409-021-01148-1,Physics-informed neural networks (PINNs) for f...,2021,2171,,"[Shengze Cai, Zhiping Mao, Zhicheng Wang, Ming...","[Discretization, Inverse problem, Flow (mathem...","[https://openalex.org/W321261355, https://open...","[https://openalex.org/W7051869546, https://ope...",physics-informed neural networks (pinns) for f...
5,https://openalex.org/W3153200540,https://doi.org/10.1115/1.4050542,Physics-Informed Neural Networks for Heat Tran...,2021,1347,Abstract Physics-informed neural networks (PIN...,"[Shengze Cai, Zhicheng Wang, Sifan Wang, Paris...","[Heat transfer, Discretization, Artificial neu...","[https://openalex.org/W321261355, https://open...","[https://openalex.org/W7169786101, https://ope...",physics-informed neural networks for heat tran...
6,https://openalex.org/W4391629689,https://doi.org/10.1016/j.cma.2024.116813,Respecting causality for training physics-info...,2024,572,,"[Sifan Wang, Shyam Sankaran, Paris G. Perdikaris]","[Causality (physics), Simple (philosophy), Lor...","[https://openalex.org/W581347801, https://open...","[https://openalex.org/W4221160766, https://ope...",respecting causality for training physics-info...
7,https://openalex.org/W3200673624,https://doi.org/10.1137/20m1318043,Understanding and Mitigating Gradient Flow Pat...,2021,2396,The widespread use of neural networks across d...,"[Sifan Wang, Yujun Teng, Paris G. Perdikaris]","[Artificial neural network, Balanced flow, Flo...","[https://openalex.org/W177735991, https://open...","[https://openalex.org/W2999467285, https://ope...",understanding and mitigating gradient flow pat...
8,https://openalex.org/W2998366519,https://doi.org/10.1016/j.cma.2019.112789,Physics-informed neural networks for high-spee...,2019,1291,,"[Zhiping Mao, Ameya D. Jagtap, George Em Karni...","[Euler equations, Conservation law, Riemann pr...","[https://openalex.org/W321261355, https://open...","[https://openalex.org/W261330859, https://open...",physics-informed neural networks for high-spee...
9,https://openalex.org/W2900369848,https://doi.org/10.1016/j.jcp.2019.05.027,Adversarial uncertainty quantification in phys...,2019,409,,"[Yibo Yang, Paris G. Perdikar

## Phase 2: Representation (Embeddings, PCA, FAISS, Graph)

In [4]:
from src.phase2_representation.embed_specter import SpecterEmbedder, DimensionalityReducer
from src.phase2_representation.faiss_index import SemanticIndex
from src.phase2_representation.citation_graph import CitationGraph

# Embeddings
embedder = SpecterEmbedder()
embeddings_raw = embedder.embed_papers(df_clean)

# PCA
reducer = DimensionalityReducer(n_components=min(16, len(df_clean) - 1))
embeddings = reducer.fit_transform(embeddings_raw)

# Citation Graph
cg = CitationGraph(directed=True)
cg.build_from_dataframe(df_clean)
cg.compute_pagerank()
cg.compute_hits()
print("Citation network constructed:", cg.graph.number_of_nodes(), "nodes,", cg.graph.number_of_edges(), "edges.")

c:\Bhavya\IITGN\Academics\7th Sem\Data Science\Project\RGD\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9757.39it/s]


Citation network constructed: 10 nodes, 15 edges.


## Phase 3: Community & Frontier Detection

In [5]:
from src.phase3_detection.semantic_cluster import SemanticCommunityDetector
from src.phase3_detection.graph_cluster import CitationCommunityDetector
from src.phase3_detection.frontier_scoring import FrontierScorer

sem_detector = SemanticCommunityDetector(n_clusters=min(3, len(df_clean)))
df_clean = sem_detector.assign_to_dataframe(df_clean, embeddings)

graph_detector = CitationCommunityDetector(method="spectral", n_clusters=min(3, len(df_clean)))
graph_detector.fit_predict(cg.graph)
df_clean = graph_detector.assign_to_dataframe(df_clean)

scorer = FrontierScorer()
frontiers = scorer.score_community_frontiers(df_clean, embeddings, cg.graph)
display(frontiers)

c:\Bhavya\IITGN\Academics\7th Sem\Data Science\Project\RGD\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


,community_a,community_b,size_a,size_b,mean_semantic_similarity,max_semantic_similarity,cross_citation_edges,citation_density,frontier_score,is_candidate_frontier
0,1,0,5,3,-0.20076,0.19131,4,0.133333,-0.173992,False


## Phase 4: Statistical Validation & Dashboard

In [6]:
from src.phase4_validation.hypothesis_test import StatisticalValidator
from src.phase4_validation.visualization import LandscapeVisualizer

validator = StatisticalValidator(n_iterations=20)
validated_frontiers = validator.bootstrap_frontier_stability(frontiers, df_clean, embeddings, cg.graph)
display(validated_frontiers)

vis = LandscapeVisualizer(output_dir="../reports")
report_path = vis.generate_html_summary_report("physics informed neural networks", df_clean, validated_frontiers)
print("Generated report:", report_path)

,community_a,community_b,size_a,size_b,mean_semantic_similarity,max_semantic_similarity,cross_citation_edges,citation_density,frontier_score,is_candidate_frontier,bootstrap_ci_95_low,bootstrap_ci_95_high,bootstrap_persistence
0,1,0,5,3,-0.20076,0.19131,4,0.133333,-0.173992,False,-0.328218,-0.104779,0.95


Generated report: ../reports\frontier_report.html
